# Fine-tuning NlpHUST/gpt2-vietnamese cho AI Service

**Model:** `NlpHUST/gpt2-vietnamese` (~125M params, ~500MB)

**Format dataset:** `Chủ đề: <topic>. Status: <text>` (VI) hoặc `Topic: <topic>. Status: <text>` (EN) — model học mapping topic → status liên quan.

**Hướng dẫn:**
1. Runtime → Change runtime type → **T4 GPU** (miễn phí)
2. Upload file `data/training_data.jsonl` mới (12,462 samples) ở cell 3
3. Chạy tuần tự các cell
4. Sau khi xong: tải `fine_tuned_model.zip` về, **XOÁ** model cũ ở `saved_models/fine_tuned/`, giải nén bản mới vào đó

**Thời gian train ước tính:** ~30–45 phút trên T4 GPU với 12.5k samples × 15 epochs (có EarlyStopping nên thường dừng sớm ở epoch 8–12).

## 1. Cài thư viện

In [ ]:
!pip install -q -U transformers accelerate datasets
print('\n>>> CHU Y: Sau khi cell nay xong, vao menu Runtime > Restart session')
print('>>> Sau do bo qua cell nay, chay tiep tu cell 2 (Kiem tra GPU)')

## 2. Kiểm tra GPU

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
else:
    print('Khong tim thay GPU. Vao Runtime > Change runtime type > T4 GPU.')

## 3. Upload file `training_data.jsonl`

File này được sinh bằng `python data/build_dataset.py` ở local. Mỗi dòng có format:
```json
{"text": "Chủ đề: đi ăn. Status: Hôm nay đi ăn bún bò ngon quá!"}
```

In [ ]:
from google.colab import files
print('Chon file training_data.jsonl tu may tinh:')
uploaded = files.upload()

## 4. Load và chia train/val

In [ ]:
import json, random

random.seed(42)

texts = []
for fname in uploaded:
    with open(fname, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                t = obj.get('text', '').strip()
                if t and len(t) >= 15:
                    texts.append(t)
            except json.JSONDecodeError:
                pass

# Dedupe
texts = list(dict.fromkeys(texts))
random.shuffle(texts)

split = max(1, int(len(texts) * 0.1))
val_texts   = texts[:split]
train_texts = texts[split:]

print(f'Tong: {len(texts)} | Train: {len(train_texts)} | Val: {len(val_texts)}')
print()
print('3 mau dau:')
for t in train_texts[:3]:
    print(f'  - {t[:100]}')

## 5. Load model + tokenizer

`NlpHUST/gpt2-vietnamese` là GPT-2 small (~125M params) pre-train trên tiếng Việt. Nhẹ hơn xglm-564M khoảng 4–5 lần, inference nhanh hơn nhiều trên CPU laptop.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'NlpHUST/gpt2-vietnamese'
print(f'Loading {MODEL_NAME} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.bos_token is None:
    tokenizer.bos_token = tokenizer.eos_token

print(f'pad_token: "{tokenizer.pad_token}" (id={tokenizer.pad_token_id})')
print(f'eos_token: "{tokenizer.eos_token}" (id={tokenizer.eos_token_id})')

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Quan trong: ensure model embeddings match tokenizer vocab.
# Mot so model (vi du NlpHUST/gpt2-vietnamese) co vocab_size cua model
# khac voi len(tokenizer) -> can resize de tranh CUDA assert.
print(f'\nBefore resize: model vocab = {model.config.vocab_size}, tokenizer = {len(tokenizer)}')
model.resize_token_embeddings(len(tokenizer))
print(f'After  resize: model vocab = {model.config.vocab_size}, tokenizer = {len(tokenizer)}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

print(f'\nLoaded on {device}.')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## 6. Tạo Dataset class

- Padding tới `MAX_SEQ_LENGTH` (96 đủ cho status mạng xã hội)
- `labels = -100` tại padding → không tính loss trên padding

In [ ]:
from torch.utils.data import Dataset

MAX_SEQ_LENGTH = 96
PAD_ID = tokenizer.pad_token_id

class StatusDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.examples = []
        for text in texts:
            enc = tokenizer(
                text, truncation=True, max_length=max_length,
                padding='max_length', return_tensors='pt',
            )
            ids = enc['input_ids'].squeeze(0)
            mask = enc['attention_mask'].squeeze(0)
            labels = ids.clone()
            labels[labels == PAD_ID] = -100
            self.examples.append({
                'input_ids': ids, 'attention_mask': mask, 'labels': labels,
            })
    def __len__(self):  return len(self.examples)
    def __getitem__(self, i): return self.examples[i]

train_dataset = StatusDataset(train_texts, tokenizer, MAX_SEQ_LENGTH)
val_dataset   = StatusDataset(val_texts,   tokenizer, MAX_SEQ_LENGTH)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

## 7. Training

Hyperparams cho model 125M:
- **15 epochs** (đủ để học pattern + có EarlyStopping bảo vệ khỏi overfit)
- batch 16, lr 3e-5, warmup 10%
- fp16 trên GPU → nhanh gấp đôi
- **EarlyStopping patience=3**: nếu val_loss không giảm trong 3 epoch liên tiếp → tự dừng (tránh overfit)

In [ ]:
import os
import math
from transformers import (
    Trainer, TrainingArguments,
    TrainerCallback, TrainerState, TrainerControl,
    EarlyStoppingCallback,
)

# Bat sync CUDA error de neu co loi se in ra dung cho (de debug)
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

EPOCHS = 15
BATCH_SIZE = 16
LR = 3e-5
EARLY_STOP_PATIENCE = 3

# T4 ho tro bf16 (on dinh hon fp16, khong bi NaN)
# Neu GPU khac (V100/P100) -> fallback ve fp32 (cham hon nhung an toan)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

# Cu the cho T4: dung fp32 cho on dinh, T4 van nhanh hon CPU 30-50x
# Neu muon nhanh hon -> doi use_bf16 = True (T4 ho tro nhung cham hon fp16)
USE_MIXED_PRECISION = False  # An toan nhat. Doi sang True neu muon nhanh hon

print(f'Mixed precision: bf16={use_bf16 and USE_MIXED_PRECISION} | fp16={use_fp16 and USE_MIXED_PRECISION}')

metrics_records = []
_current = {}

steps_per_epoch = math.ceil(len(train_dataset) / BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(10, int(total_steps * 0.10))
print(f'steps/epoch={steps_per_epoch} | total_steps={total_steps} | warmup={warmup_steps}')

class MetricsCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kw):
        print(f'\n--- Epoch {int(state.epoch or 0)+1}/{int(args.num_train_epochs)} ---', flush=True)
    def on_log(self, args, state, control, logs=None, **kw):
        global _current
        if not logs: return
        if 'loss' in logs:
            loss = float(logs['loss'])
            _current = {
                'epoch': round(state.epoch or 0, 2),
                'step': state.global_step,
                'train_loss': round(loss, 4),
                'train_perplexity': round(math.exp(min(loss, 20)), 2),
            }
            print(f'  step {state.global_step:>4} | loss={loss:.4f} | ppl={math.exp(min(loss,20)):.2f}', flush=True)
        if 'eval_loss' in logs:
            el = float(logs['eval_loss'])
            rec = {
                **_current,
                'epoch': round(state.epoch or 0, 1),
                'eval_loss': round(el, 4),
                'eval_perplexity': round(math.exp(min(el,20)), 2),
            }
            metrics_records.append(rec)
            print(f'  EVAL epoch {rec["epoch"]:.0f} | train={rec.get("train_loss","?")} | eval={rec["eval_loss"]} | ppl={rec["eval_perplexity"]}', flush=True)

args = TrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    max_grad_norm=1.0,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_steps=max(1, steps_per_epoch // 4),
    save_total_limit=2,
    report_to='none',
    fp16=use_fp16 and USE_MIXED_PRECISION,
    bf16=use_bf16 and USE_MIXED_PRECISION,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[
        MetricsCallback(),
        EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE),
    ],
)

print('\nBat dau training ...')
trainer.train()
print('\nTraining xong.')

## 8. Bảng metrics từng epoch

In [ ]:
with open('metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_records, f, ensure_ascii=False, indent=2)

print(f"{'Epoch':>6}  {'Train Loss':>11}  {'Eval Loss':>10}  {'Train PPL':>10}  {'Eval PPL':>9}")
print('-' * 55)
for r in metrics_records:
    print(f"{r.get('epoch','?'):>6}  {r.get('train_loss',0):>11.4f}  "
          f"{r.get('eval_loss',0):>10.4f}  {r.get('train_perplexity',0):>10.2f}  "
          f"{r.get('eval_perplexity',0):>9.2f}")

## 9. Biểu đồ Loss & Perplexity

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

epochs     = [r['epoch'] for r in metrics_records]
train_loss = [r.get('train_loss') for r in metrics_records]
eval_loss  = [r.get('eval_loss')  for r in metrics_records]
train_ppl  = [r.get('train_perplexity') for r in metrics_records]
eval_ppl   = [r.get('eval_perplexity')  for r in metrics_records]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(epochs, train_loss, marker='o', label='Train Loss',      color='#2196F3')
axes[0].plot(epochs, eval_loss,  marker='s', label='Validation Loss', color='#F44336')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend(); axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

axes[1].plot(epochs, train_ppl, marker='o', label='Train Perplexity',      color='#4CAF50')
axes[1].plot(epochs, eval_ppl,  marker='s', label='Validation Perplexity', color='#FF9800')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Perplexity')
axes[1].set_title('Training & Validation Perplexity')
axes[1].legend(); axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('training_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Bieu do da luu: training_charts.png')

## 10. Test thử model sau khi train

Sinh thử với prompt `Chủ đề: đi ăn. Status:` xem model có ra đúng chủ đề chưa.

In [ ]:
model.eval()

TEST_PROMPTS = [
    'Chủ đề: đi ăn. Status:',
    'Chủ đề: cà phê sáng. Status:',
    'Chủ đề: du lịch Đà Lạt. Status:',
    'Chủ đề: gia đình. Status:',
    'Chủ đề: gym. Status:',
    'Chủ đề: tâm trạng buồn. Status:',
    'Chủ đề: mùa thu Hà Nội. Status:',
    'Topic: food. Status:',
    'Topic: coffee. Status:',
    'Topic: travel. Status:',
    'Topic: gym. Status:',
    'Topic: motivation. Status:',
]

for prompt in TEST_PROMPTS:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    prompt_len = inputs['input_ids'].shape[1]
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=35,
            min_new_tokens=8,
            num_return_sequences=3,
            do_sample=True,
            temperature=0.85,
            top_k=50,
            top_p=0.92,
            repetition_penalty=1.25,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
        )
    print(f'\n>>> {prompt}')
    for i in range(out.shape[0]):
        text = tokenizer.decode(out[i][prompt_len:], skip_special_tokens=True)
        text = text.split('\n')[0].split('Chủ đề:')[0].split('Topic:')[0].strip()
        print(f'   {i+1}. {text}')

## 11. Lưu và tải về máy

In [ ]:
import shutil
from google.colab import files

SAVE_DIR = './fine_tuned'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
shutil.copy('metrics.json', SAVE_DIR + '/metrics.json')
print(f'Model saved to {SAVE_DIR}')

shutil.make_archive('fine_tuned_model', 'zip', SAVE_DIR)
print('Da nen: fine_tuned_model.zip')

files.download('fine_tuned_model.zip')
files.download('metrics.json')
files.download('training_charts.png')
print('Tai xuong xong!')

## Sau khi tải về máy

```
1. Giai nen fine_tuned_model.zip
2. Copy noi dung vao: ai-service/saved_models/fine_tuned/
3. Copy metrics.json    vao: ai-service/logs/metrics_colab.json
4. Khoi dong lai service: python run.py
5. Test:  POST http://localhost:8092/ai/suggest
          { "context": "đi ăn", "mood": "funny", "language": "vi", "num_suggestions": 5 }
6. Check: GET  http://localhost:8092/ai/model/info   →  fine_tuned: true
```